In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2013_2014"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "SMQ_H.xpt"
csv_file = DATA_DIR / "SMQ_H_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Completely untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2013-2014 SMQ_H XPT -> CSV")
print("=" * 90)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SMQ020",
    "SMD030",
    "SMQ040",
    "SMQ050Q",
    "SMQ050U",
    "SMD055",
    "SMD057",
    "SMQ078",
    "SMD641",
    "SMD650",
    "SMD093",
    "SMDUPCA",
    "SMD100BR",
    "SMD100FL",
    "SMD100MN",
    "SMD100LN",
    "SMD100TR",
    "SMD100NI",
    "SMD100CO",
    "SMQ621",
    "SMD630",
    "SMQ661",
    "SMQ665A",
    "SMQ665B",
    "SMQ665C",
    "SMQ665D",
    "SMQ670",
    "SMQ848",
    "SMQ852Q",
    "SMQ852U",
    "SMAQUEX2"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "SMQ_H columns/order differ.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 7168:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 32:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 7,168 rows x 32 columns."
)


# ============================================================
# 4. DEFINE FIELD TYPES
# ============================================================

#
# SMQ_H IS A MIXED-TYPE DATASET.
#
#
# TEXT:
#
# SMDUPCA
#     12-digit cigarette UPC.
#     MUST remain text because leading zeros matter.
#
# SMD100BR
#     Cigarette brand/sub-brand.
#
#
# GENUINE DECIMAL:
#
# SMD100NI
#     FTC nicotine content.
#     CDC range = 0.1 to 2.0.
#
#
# REMAINING NUMERIC FIELDS:
#
# Whole-number questionnaire codes / quantities.
#

STRING_COLUMNS = [
    "SMDUPCA",
    "SMD100BR"
]


DECIMAL_COLUMNS = [
    "SMD100NI"
]


NUMERIC_COLUMNS = [
    col
    for col in EXPECTED_COLUMNS
    if col not in STRING_COLUMNS
]


INTEGER_COLUMNS = [
    col
    for col in NUMERIC_COLUMNS
    if col not in DECIMAL_COLUMNS
]


if len(STRING_COLUMNS) != 2:

    raise ValueError(
        "Expected 2 string fields."
    )


if len(DECIMAL_COLUMNS) != 1:

    raise ValueError(
        "Expected exactly one decimal field."
    )


if len(INTEGER_COLUMNS) != 29:

    raise ValueError(
        "Expected 29 whole-number numeric fields."
    )


print("\n--- FIELD TYPE PLAN ---")

print(
    "String fields:",
    STRING_COLUMNS
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number numeric fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK STRING VARIABLES BEFORE ANY CLEANING
# ============================================================

print("\n--- ORIGINAL TEXT FIELD CHECK ---")


for col in STRING_COLUMNS:

    values = (
        df[col]
        .fillna("")
        .astype(str)
    )

    print(
        f"{col} nonblank:",
        f"{int(values.ne('').sum()):,}"
    )


# ============================================================
# 6. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT / pandas representation:
#
#     5.397605346934028e-79
#
# NEVER globally replace before validating location/count.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


loaded_numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if loaded_numeric_cols != NUMERIC_COLUMNS:

    raise ValueError(
        "Numeric field structure differs.\n"
        f"Expected:\n{NUMERIC_COLUMNS}\n\n"
        f"Actual:\n{loaded_numeric_cols}"
    )


tiny_counts = (
    df[NUMERIC_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[NUMERIC_COLUMNS]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)

print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


print("\nTiny artifacts by variable:")

print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 7. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Verified against newly attached SMQ_H copy:
#
# SMD030    = 120
# SMD641    =  68
# SMD100FL  =  12
# SMD100MN  = 671
# SMQ852Q   =   3
#
# TOTAL     = 874
#

EXPECTED_TINY_COUNTS = {
    "SMD030": 120,
    "SMD641": 68,
    "SMD100FL": 12,
    "SMD100MN": 671,
    "SMQ852Q": 3
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{actual_tiny_counts}"
    )


if tiny_total != 874:

    raise ValueError(
        f"Unexpected total tiny count: {tiny_total:,}"
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary zeros existed "
        "before correction."
    )


print(
    "PASS: Exact 874-value tiny-zero pattern validated."
)


# ============================================================
# 8. RESTORE ONLY VALIDATED ZEROS
# ============================================================

#
# DO NOT:
#
#     df.replace(TINY_VALUE, 0)
#
# globally.
#

print("\n--- VALIDATED ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected {expected_count:,}, "
            f"found {actual_count:,}."
        )


    df.loc[
        mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 874:

    raise ValueError(
        "Incorrect total restoration count."
    )


print(
    "PASS:",
    f"{correction_total:,}",
    "validated zeros restored."
)


# ============================================================
# 9. CREATE EXPECTED CORRECTED SOURCE
# ============================================================

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 10. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[NUMERIC_COLUMNS]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print(
    "\nTiny values remaining:",
    remaining_tiny
)


if remaining_tiny != 0:

    raise ValueError(
        "Tiny values remain after correction."
    )


print(
    "PASS: No tiny XPORT artifacts remain."
)


# ============================================================
# 11. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# MUST occur before any Int64 conversion.
#
# np.round() is used ONLY for validation.
# It does NOT modify data.
#

print("\n--- GENUINE DECIMAL DETECTION ---")


detected_decimal_columns = []


for col in NUMERIC_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    if fractional_count > 0:

        detected_decimal_columns.append(
            col
        )


        print(
            f"{col}: "
            f"{fractional_count:,} fractional observations"
        )


if detected_decimal_columns != DECIMAL_COLUMNS:

    raise ValueError(
        "Decimal-variable pattern differs.\n"
        f"Expected: {DECIMAL_COLUMNS}\n"
        f"Actual:   {detected_decimal_columns}"
    )


print(
    "PASS: SMD100NI is the only genuine "
    "numeric decimal field."
)


# ============================================================
# 12. VALIDATE SMD100NI
# ============================================================

s = df["SMD100NI"]


nicotine_nonmissing = int(
    s.notna()
    .sum()
)


nicotine_missing = int(
    s.isna()
    .sum()
)


nicotine_min = float(
    s.min()
)


nicotine_max = float(
    s.max()
)


nicotine_values = (
    s.dropna()
    .to_numpy(
        dtype=float
    )
)


nicotine_fractional = int(
    (
        ~np.isclose(
            nicotine_values,
            np.round(
                nicotine_values
            ),
            rtol=0,
            atol=1e-12
        )
    )
    .sum()
)


nicotine_whole_looking = (
    nicotine_nonmissing
    -
    nicotine_fractional
)


print("\n--- SMD100NI ---")

print(
    "Nonmissing:",
    nicotine_nonmissing
)

print(
    "Missing:",
    nicotine_missing
)

print(
    "Range:",
    nicotine_min,
    "to",
    nicotine_max
)

print(
    "Genuinely fractional:",
    nicotine_fractional
)

print(
    "Whole-looking values:",
    nicotine_whole_looking
)


if nicotine_nonmissing != 828:

    raise ValueError(
        "Unexpected SMD100NI nonmissing count."
    )


if nicotine_missing != 6340:

    raise ValueError(
        "Unexpected SMD100NI missing count."
    )


if nicotine_min != 0.1:

    raise ValueError(
        "Unexpected nicotine minimum."
    )


if nicotine_max != 2.0:

    raise ValueError(
        "Unexpected nicotine maximum."
    )


if nicotine_fractional != 629:

    raise ValueError(
        "Unexpected fractional nicotine count."
    )


print(
    "PASS: SMD100NI decimal fidelity structure validated."
)


# ============================================================
# 13. CLEAN AND VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(
        seqn_values
    ),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 73557:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 83729:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Missing SEQN found."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN found."
    )


print(
    "PASS: SEQN 73557-83729 validated."
)


# ============================================================
# 14. CONVERT ONLY PROVEN WHOLE-NUMBER FIELDS
# ============================================================

#
# Convert:
#
#     29 integer/code fields
#
# DO NOT convert:
#
#     SMD100NI
#
# DO NOT alter:
#
#     SMDUPCA
#     SMD100BR
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(
                values
            ),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )


    #
    # NO .round()
    #
    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: 29 whole-number fields converted to Int64."
)

print(
    "PASS: SMD100NI left unchanged as float."
)

print(
    "PASS: UPC and brand fields remain strings."
)


# ============================================================
# 15. CDC CHECK - SMD030
# ============================================================

#
# 0 = Never smoked cigarettes regularly.
#

s = df["SMD030"]


if int(
    s.between(
        7,
        64
    )
    .sum()
) != 2448:

    raise ValueError(
        "SMD030 7-64 count differs."
    )


if int(
    s.eq(0)
    .sum()
) != 120:

    raise ValueError(
        "SMD030 zero count differs."
    )


if int(
    s.eq(999)
    .sum()
) != 11:

    raise ValueError(
        "SMD030 999 count differs."
    )


if int(
    s.isna()
    .sum()
) != 4589:

    raise ValueError(
        "SMD030 missing count differs."
    )


print(
    "PASS: SMD030 matches CDC."
)


# ============================================================
# 16. CDC CHECK - SMD641
# ============================================================

#
# Number of days smoked in past 30 days.
#
# Allowed:
#
#     0-30
#

if int(
    df["SMD641"]
    .between(
        0,
        30
    )
    .sum()
) != 1316:

    raise ValueError(
        "SMD641 0-30 count differs."
    )


if int(
    df["SMD641"]
    .eq(0)
    .sum()
) != 68:

    raise ValueError(
        "SMD641 zero count differs."
    )


if int(
    df["SMD641"]
    .isna()
    .sum()
) != 5852:

    raise ValueError(
        "SMD641 missing count differs."
    )


print(
    "PASS: SMD641 validated."
)


# ============================================================
# 17. CDC CHECK - SMD100FL
# ============================================================

#
# 0 Non-filter
# 1 Filter
#

filter_counts = (
    df["SMD100FL"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if filter_counts != {
    0: 12,
    1: 1079
}:

    raise ValueError(
        "SMD100FL frequencies differ."
    )


if int(
    df["SMD100FL"]
    .isna()
    .sum()
) != 6077:

    raise ValueError(
        "SMD100FL missing differs."
    )


print(
    "PASS: SMD100FL validated."
)


# ============================================================
# 18. CDC CHECK - SMD100MN
# ============================================================

#
# 0 Non-menthol
# 1 Menthol
#

menthol_counts = (
    df["SMD100MN"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if menthol_counts != {
    0: 671,
    1: 421
}:

    raise ValueError(
        "SMD100MN frequencies differ."
    )


if int(
    df["SMD100MN"]
    .isna()
    .sum()
) != 6076:

    raise ValueError(
        "SMD100MN missing differs."
    )


print(
    "PASS: SMD100MN validated."
)


# ============================================================
# 19. CDC CHECK - TAR / NICOTINE / CO
# ============================================================

#
# SMD100TR:
#     2-24, n=828
#
# SMD100NI:
#     0.1-2.0, n=828
#
# SMD100CO:
#     3-19, n=828
#

if int(
    df["SMD100TR"]
    .between(
        2,
        24
    )
    .sum()
) != 828:

    raise ValueError(
        "SMD100TR differs."
    )


if int(
    df["SMD100NI"]
    .between(
        0.1,
        2.0
    )
    .sum()
) != 828:

    raise ValueError(
        "SMD100NI differs."
    )


if int(
    df["SMD100CO"]
    .between(
        3,
        19
    )
    .sum()
) != 828:

    raise ValueError(
        "SMD100CO differs."
    )


for col in [
    "SMD100TR",
    "SMD100NI",
    "SMD100CO"
]:

    if int(
        df[col]
        .isna()
        .sum()
    ) != 6340:

        raise ValueError(
            f"{col} missing count differs."
        )


print(
    "PASS: Tar/nicotine/CO ranges validated."
)


# ============================================================
# 20. TAR/NICOTINE/CO MISSING MASK
# ============================================================

tar_present = (
    df["SMD100TR"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


nic_present = (
    df["SMD100NI"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


co_present = (
    df["SMD100CO"]
    .notna()
    .to_numpy(
        dtype=bool
    )
)


if int(
    np.sum(
        tar_present
        !=
        nic_present
    )
) != 0:

    raise ValueError(
        "Tar and nicotine masks differ."
    )


if int(
    np.sum(
        nic_present
        !=
        co_present
    )
) != 0:

    raise ValueError(
        "Nicotine and CO masks differ."
    )


print(
    "PASS: Tar/nicotine/CO masks match exactly."
)


# ============================================================
# 21. VALIDATE SMDUPCA STRING STRUCTURE
# ============================================================

#
# IMPORTANT:
#
# DO NOT convert UPC to numeric.
#
# Leading zeros are real data.
#

upc = (
    df["SMDUPCA"]
    .fillna("")
    .astype(str)
)


valid_upc_mask = (
    upc.str.fullmatch(
        r"\d{12}"
    )
)


valid_upc_count = int(
    valid_upc_mask.sum()
)


no_match_count = int(
    upc.eq(
        "* NO MATCH *"
    )
    .sum()
)


blank_upc_count = int(
    upc.eq("")
    .sum()
)


leading_zero_upc = int(
    (
        valid_upc_mask
        &
        upc.str.startswith("0")
    )
    .sum()
)


print("\n--- SMDUPCA ---")

print(
    "Valid 12-digit UPC:",
    valid_upc_count
)

print(
    "* NO MATCH *:",
    no_match_count
)

print(
    "Blank:",
    blank_upc_count
)

print(
    "Leading-zero UPC:",
    leading_zero_upc
)


if valid_upc_count != 758:

    raise ValueError(
        "Unexpected valid UPC count."
    )


if no_match_count != 58:

    raise ValueError(
        "Unexpected NO MATCH count."
    )


if blank_upc_count != 6352:

    raise ValueError(
        "Unexpected blank UPC count."
    )


if leading_zero_upc != 722:

    raise ValueError(
        "Unexpected leading-zero UPC count."
    )


print(
    "PASS: SMDUPCA structure validated."
)


# ============================================================
# 22. VALIDATE SMD100BR TEXT
# ============================================================

brand = (
    df["SMD100BR"]
    .fillna("")
    .astype(str)
)


brand_nonblank = int(
    brand.ne("")
    .sum()
)


brand_blank = int(
    brand.eq("")
    .sum()
)


if brand_nonblank != 1092:

    raise ValueError(
        "Unexpected populated brand count."
    )


if brand_blank != 6076:

    raise ValueError(
        "Unexpected blank brand count."
    )


print(
    "PASS: SMD100BR text counts validated."
)


# ============================================================
# 23. CDC CHECK - SMQ852Q
# ============================================================

#
# Range:
#
#     0-356
#
# Plus:
#
#     7777 Refused
#     9999 Don't know
#
# Attached XPT contains exactly 3 valid zeros.
#

s = df["SMQ852Q"]


if int(
    s.between(
        0,
        356
    )
    .sum()
) != 577:

    raise ValueError(
        "SMQ852Q range count differs."
    )


if int(
    s.eq(0)
    .sum()
) != 3:

    raise ValueError(
        "SMQ852Q zero count differs."
    )


if int(
    s.eq(7777)
    .sum()
) != 1:

    raise ValueError(
        "SMQ852Q refused count differs."
    )


if int(
    s.eq(9999)
    .sum()
) != 2:

    raise ValueError(
        "SMQ852Q don't-know count differs."
    )


if int(
    s.isna()
    .sum()
) != 6588:

    raise ValueError(
        "SMQ852Q missing count differs."
    )


print(
    "PASS: SMQ852Q validated."
)


# ============================================================
# 24. CDC CHECK - SMQ852U
# ============================================================

smq852u_counts = (
    df["SMQ852U"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if smq852u_counts != {
    1: 281,
    2: 138,
    3: 158
}:

    raise ValueError(
        "SMQ852U frequencies differ."
    )


if int(
    df["SMQ852U"]
    .isna()
    .sum()
) != 6591:

    raise ValueError(
        "SMQ852U missing differs."
    )


print(
    "PASS: SMQ852U validated."
)


# ============================================================
# 25. CDC CHECK - SMAQUEX2
# ============================================================

#
# 1 = Home Interview age 18+
#
# 2 = ACASI age 12-17
#

mode_counts = (
    df["SMAQUEX2"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if mode_counts != {
    1: 6113,
    2: 1055
}:

    raise ValueError(
        "SMAQUEX2 frequencies differ."
    )


if int(
    df["SMAQUEX2"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "Unexpected missing SMAQUEX2."
    )


print(
    "PASS: SMAQUEX2 validated."
)


# ============================================================
# 26. FINAL ZERO PATTERN
# ============================================================

#
# Before repair:
#
#     ordinary numeric zeros = 0
#
# Therefore all zeros after repair should exactly equal
# the validated artifact locations.
#

final_zero_counts = {}


for col in NUMERIC_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )


    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")

print(
    final_zero_counts
)


if final_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{final_zero_counts}"
    )


if sum(
    final_zero_counts.values()
) != 874:

    raise ValueError(
        "Unexpected final zero total."
    )


print(
    "PASS: Final zero pattern exactly matches "
    "all 874 validated restorations."
)


# ============================================================
# 27. NUMERIC FIDELITY BEFORE EXPORT
# ============================================================

#
# ONLY the validated tiny -> 0 values may differ from
# the original pandas-decoded XPT.
#

numeric_fidelity_failures = []


for col in NUMERIC_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        numeric_fidelity_failures.append(
            col
        )


if numeric_fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes:\n"
        + str(
            numeric_fidelity_failures
        )
    )


print(
    "PASS: All 30 numeric fields match "
    "the corrected source exactly."
)


# ============================================================
# 28. TEXT FIDELITY BEFORE EXPORT
# ============================================================

for col in STRING_COLUMNS:

    original = (
        source_df[col]
        .fillna("")
        .astype(str)
        .to_numpy()
    )


    current = (
        df[col]
        .fillna("")
        .astype(str)
        .to_numpy()
    )


    if not np.array_equal(
        original,
        current
    ):

        raise ValueError(
            f"{col} changed before export."
        )


print(
    "PASS: SMDUPCA and SMD100BR "
    "unchanged before export."
)


# ============================================================
# 29. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO:
#
#     round()
#
# NO:
#
#     float_format=
#
# NO:
#
#     numeric conversion of UPC
#
#
# This preserves:
#
# SMD100NI
#     genuine decimal values AS IS
#
# SMDUPCA
#     leading-zero UPCs
#
# SMD100BR
#     exact text
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 30. RE-READ NUMERIC CSV WITH ROUND-TRIP PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column structure changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 31. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

csv_numeric_failures = []


for col in NUMERIC_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            roundtrip_df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    if not np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        csv_numeric_failures.append(
            col
        )


if csv_numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences:\n"
        + str(
            csv_numeric_failures
        )
    )


print(
    "PASS: 0 numeric differences across "
    "all 30 numeric fields."
)


# ============================================================
# 32. NUMERIC MISSING-VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in NUMERIC_COLUMNS:

    source_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    csv_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            source_missing
            !=
            csv_missing
        )
    )


print(
    "\nMissing-position differences:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Numeric missing positions changed."
    )


print(
    "PASS: Numeric missing positions preserved."
)


# ============================================================
# 33. RE-READ FINAL CSV AS RAW TEXT
# ============================================================

#
# Required for UPC/text formatting checks.
#

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 34. EXACT UPC FIDELITY AFTER EXPORT
# ============================================================

source_upc = (
    source_df["SMDUPCA"]
    .fillna("")
    .astype(str)
    .to_numpy()
)


csv_upc = (
    raw_csv_df["SMDUPCA"]
    .to_numpy()
)


if not np.array_equal(
    source_upc,
    csv_upc
):

    raise ValueError(
        "SMDUPCA changed during CSV export."
    )


print(
    "PASS: SMDUPCA preserved character-for-character."
)


# ============================================================
# 35. EXACT BRAND FIDELITY AFTER EXPORT
# ============================================================

source_brand = (
    source_df["SMD100BR"]
    .fillna("")
    .astype(str)
    .to_numpy()
)


csv_brand = (
    raw_csv_df["SMD100BR"]
    .to_numpy()
)


if not np.array_equal(
    source_brand,
    csv_brand
):

    raise ValueError(
        "SMD100BR changed during export."
    )


print(
    "PASS: SMD100BR preserved character-for-character."
)


# ============================================================
# 36. FINAL UPC STRUCTURE CHECK
# ============================================================

final_upc = raw_csv_df["SMDUPCA"]


final_valid_upc = int(
    final_upc
    .str.fullmatch(
        r"\d{12}"
    )
    .sum()
)


final_no_match = int(
    final_upc
    .eq(
        "* NO MATCH *"
    )
    .sum()
)


final_blank_upc = int(
    final_upc
    .eq("")
    .sum()
)


final_leading_zero = int(
    (
        final_upc
        .str.fullmatch(
            r"\d{12}"
        )
        &
        final_upc
        .str.startswith("0")
    )
    .sum()
)


if final_valid_upc != 758:

    raise ValueError(
        "Valid UPC count changed."
    )


if final_no_match != 58:

    raise ValueError(
        "NO MATCH count changed."
    )


if final_blank_upc != 6352:

    raise ValueError(
        "Blank UPC count changed."
    )


if final_leading_zero != 722:

    raise ValueError(
        "Leading-zero UPC count changed."
    )


print(
    "PASS: All UPC formatting preserved."
)


# ============================================================
# 37. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in NUMERIC_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values.eq(
            TINY_VALUE
        )
        .sum()
    )


print(
    "\nTiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain."
    )


print(
    "PASS: No tiny artifacts in CSV."
)


# ============================================================
# 38. .0 CHECK - INTEGER FIELDS ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# DO NOT check SMD100NI here.
#
# SMD100NI is genuinely decimal.
#

dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


print(
    "\n--- INTEGER .0 CHECK ---"
)


print(
    dot_zero_errors
)


if dot_zero_errors:

    raise ValueError(
        "Unnecessary '.0' found in integer fields:\n"
        + str(
            dot_zero_errors
        )
    )


print(
    "PASS: No unnecessary '.0' in "
    "the 29 integer fields."
)


# ============================================================
# 39. LEGITIMATE SMD100NI .0 TOKENS
# ============================================================

#
# INFORMATIONAL ONLY.
#
# Because SMD100NI is a genuine decimal column,
# values such as:
#
#     1.0
#     2.0
#
# are legitimate.
#

nicotine_dot_zero = int(
    (
        raw_csv_df["SMD100NI"]
        .ne("")
        &
        raw_csv_df["SMD100NI"]
        .str.endswith(".0")
    )
    .sum()
)


print(
    "\nSMD100NI legitimate '.0' values:",
    nicotine_dot_zero
)


if nicotine_dot_zero != 199:

    raise ValueError(
        "Unexpected SMD100NI '.0' token count."
    )


print(
    "PASS: 199 legitimate whole-looking nicotine "
    "values preserved."
)


# ============================================================
# 40. EXACT SMD100NI DECIMAL FIDELITY
# ============================================================

expected_nicotine = (
    expected_df["SMD100NI"]
    .to_numpy(
        dtype=float
    )
)


exported_nicotine = (
    roundtrip_df["SMD100NI"]
    .to_numpy(
        dtype=float
    )
)


if not np.allclose(
    expected_nicotine,
    exported_nicotine,
    rtol=0,
    atol=0,
    equal_nan=True
):

    raise ValueError(
        "SMD100NI decimal fidelity failed."
    )


print(
    "PASS: Every SMD100NI value preserved exactly."
)


# ============================================================
# 41. FINAL CSV ZERO PATTERN
# ============================================================

csv_zero_counts = {}


for col in NUMERIC_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    zero_count = int(
        values.eq(0)
        .sum()
    )


    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print(
    "\n--- FINAL CSV ZERO PATTERN ---"
)


print(
    csv_zero_counts
)


if csv_zero_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "Final CSV zero pattern differs.\n\n"
        f"Expected:\n{EXPECTED_TINY_COUNTS}\n\n"
        f"Actual:\n{csv_zero_counts}"
    )


print(
    "PASS: Final zero pattern exactly matches "
    "the 874 validated corrections."
)


# ============================================================
# 42. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in NUMERIC_COLUMNS:

    tokens = raw_csv_df[col]


    scientific_count += int(
        tokens
        .str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 43. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 73557:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 83729:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Duplicate SEQN in CSV."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "SEQN still contains '.0'."
    )


print(
    "PASS: SEQN valid and contains no '.0'."
)


# ============================================================
# 44. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "SMQ_H CONVERSION COMPLETE"
)

print(
    "=" * 90
)

print(
    "Cycle: 2013-2014"
)

print(
    "Component: Questionnaire - Smoking, Cigarette Use"
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)

print(
    "Original tiny artifacts:",
    f"{tiny_total:,}"
)

print(
    "SMD030 zero restorations:",
    EXPECTED_TINY_COUNTS["SMD030"]
)

print(
    "SMD641 zero restorations:",
    EXPECTED_TINY_COUNTS["SMD641"]
)

print(
    "SMD100FL zero restorations:",
    EXPECTED_TINY_COUNTS["SMD100FL"]
)

print(
    "SMD100MN zero restorations:",
    EXPECTED_TINY_COUNTS["SMD100MN"]
)

print(
    "SMQ852Q zero restorations:",
    EXPECTED_TINY_COUNTS["SMQ852Q"]
)

print(
    "Total validated zero restorations:",
    correction_total
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)

print(
    "String fields:",
    STRING_COLUMNS
)

print(
    "Whole-number numeric fields:",
    len(INTEGER_COLUMNS)
)

print(
    "SMD100NI nonmissing:",
    nicotine_nonmissing
)

print(
    "SMD100NI fractional values:",
    nicotine_fractional
)

print(
    "Valid 12-digit UPC:",
    final_valid_upc
)

print(
    "Leading-zero UPC preserved:",
    final_leading_zero
)

print(
    "Brand strings:",
    brand_nonblank
)

print(
    "XPT -> CSV numeric differences: 0 expected"
)

print(
    "Missing-position differences: 0 expected"
)

print(
    "UPC differences: 0 expected"
)

print(
    "Brand-text differences: 0 expected"
)

print(
    "SMD100NI decimal differences: 0 expected"
)

print(
    "Tiny values remaining: 0 expected"
)

print(
    "Unnecessary '.0' in integer fields: 0 expected"
)

print(
    "Legitimate SMD100NI '.0' values: 199"
)

print(
    "Scientific notation: 0 expected"
)

NHANES 2013-2014 SMQ_H XPT -> CSV
Rows: 7,168
Columns: 32
PASS: Structure = 7,168 rows x 32 columns.

--- FIELD TYPE PLAN ---
String fields: ['SMDUPCA', 'SMD100BR']
Genuine decimal fields: ['SMD100NI']
Whole-number numeric fields: 29

--- ORIGINAL TEXT FIELD CHECK ---
SMDUPCA nonblank: 816
SMD100BR nonblank: 1,092

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros: 0
Tiny XPORT artifacts: 874

Tiny artifacts by variable:
SMD100MN    671
SMD030      120
SMD641       68
SMD100FL     12
SMQ852Q       3
PASS: Exact 874-value tiny-zero pattern validated.

--- VALIDATED ZERO RESTORATION ---
SMD030: 120 restored to 0
SMD641: 68 restored to 0
SMD100FL: 12 restored to 0
SMD100MN: 671 restored to 0
SMQ852Q: 3 restored to 0
PASS: 874 validated zeros restored.

Tiny values remaining: 0
PASS: No tiny XPORT artifacts remain.

--- GENUINE DECIMAL DETECTION ---
SMD100NI: 629 fractional observations
PASS: SMD100NI is the only genuine numeric decimal field.

--- SMD100NI ---
Nonmissing: 828
Missing